<a href="https://colab.research.google.com/github/Mayankrawat029/re-startdemo/blob/main/Pbl_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit pyngrok agno google-generativeai pillow duckduckgo-search pydicom

In [ ]:
!pip install kaggle

In [ ]:
!pip install ddgs

In [34]:
import zipfile

zip_path = "/content/chest-xray-pneumonia.zip"   # apni file ka naam yaha daal

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/dataset")

print("✅ Extract ho gaya")

✅ Extract ho gaya


In [35]:
!ls /content/dataset

chest_xray


In [36]:
!ls /content/dataset/chest_xray

chest_xray  __MACOSX  test  train  val


In [37]:
train_path = "/content/dataset/chest_xray/train"
test_path = "/content/dataset/chest_xray/test"

In [38]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

test_data = test_datagen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

print("✅ Dataset Loaded")

Found 5216 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
✅ Dataset Loaded


In [41]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

model = Sequential([
    Conv2D(32,(3,3),activation='relu',input_shape=(224,224,3)),
    MaxPooling2D(2,2),

    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128,activation='relu'),
    Dense(1,activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.fit(train_data, validation_data=test_data, epochs=2)

print("✅ Model Trained")

Epoch 1/2
163/163 ━━━━━━━━━━━━━━━━━━━━ 620s 4s/step - accuracy: 0.8175 - loss: 0.4878 - val_accuracy: 0.7933 - val_loss: 0.4694
Epoch 2/2
163/163 ━━━━━━━━━━━━━━━━━━━━ 632s 4s/step - accuracy: 0.9057 - loss: 0.2211 - val_accuracy: 0.7676 - val_loss: 0.5800
✅ Model Trained


In [43]:
model.save("/content/medical_model.h5")
print("✅ Model Saved")

✅ Model Saved


In [47]:
import os
from google.colab import drive

# Ensure the mount point is clean before mounting
if os.path.exists('/content/drive'):
    !rm -rf /content/drive
    print("Removed existing /content/drive directory.")

drive.mount('/content/drive', force_remount=True)
print("Drive mounted successfully.")

Removed existing /content/drive directory.
Mounted at /content/drive
Drive mounted successfully.


In [48]:
model.save("/content/drive/MyDrive/medical_model.h5")

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/content/drive/MyDrive/medical_model.h5")

print("✅ Model loaded instantly")

In [49]:
import os
print(os.listdir("/content"))

['.config', 'temp.png', 'dataset', 'app.py', 'medical_model.h5', 'drive', 'chest-xray-pneumonia.zip', 'sample_data']


In [52]:
import numpy as np
import cv2
from PIL import Image

img = Image.open("/content/dataset/chest_xray/test/NORMAL/IM-0001-0001.jpeg").convert("RGB")
img = np.array(img)
img = cv2.resize(img,(224,224))
img = img/255.0
img = np.reshape(img,(1,224,224,3))

pred = model.predict(img)

print("Prediction:", pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
Prediction: [[0.38637036]]


In [ ]:
%%writefile app.py

In [99]:
%%writefile app.py
"""
╔══════════════════════════════════════════════════════════╗
║         AI Medical Imaging Diagnosis System              ║
║         Production-Grade Streamlit Application          ║
╚══════════════════════════════════════════════════════════╝
"""

import streamlit as st
import numpy as np
import cv2
from tensorflow.keras.models import load_model, Model
from PIL import Image
import datetime
import io
import time
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ─────────────────────────────────────────────────────────────
# PAGE CONFIG  (must be first Streamlit call)
# ─────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="VisionDx AI — Pneumonia Detection",
    page_icon="🔬",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ─────────────────────────────────────────────────────────────
# CUSTOM CSS  — refined clinical dark theme
# ─────────────────────────────────────────────────────────────
st.markdown("""
<style>
/* ── Google Fonts ─────────────────────────────────────────── */
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:ital,wght@0,300;0,400;0,500;0,600;0,700;1,400&family=DM+Mono:wght@400;500&family=Playfair+Display:wght@700&display=swap');

/* ── Root tokens ──────────────────────────────────────────── */
:root {
    --bg-0:   #0a0e1a;
    --bg-1:   #111827;
    --bg-2:   #1a2235;
    --bg-3:   #1f2d42;
    --border: rgba(99,180,255,0.12);
    --accent: #3b9eff;
    --accent2:#00d4aa;
    --danger: #ff4e6a;
    --warn:   #ffb347;
    --txt-1:  #e8edf5;
    --txt-2:  #8fa3be;
    --txt-3:  #5c7490;
    --radius: 14px;
    --shadow: 0 8px 32px rgba(0,0,0,0.45);
}

/* ── Global reset ─────────────────────────────────────────── */
html, body, [class*="css"] {
    font-family: 'DM Sans', sans-serif;
    background-color: var(--bg-0);
    color: var(--txt-1);
}

/* ── Main header ──────────────────────────────────────────── */
.app-header {
    display: flex;
    align-items: center;
    gap: 18px;
    padding: 28px 0 20px;
    border-bottom: 1px solid var(--border);
    margin-bottom: 28px;
}
.app-header .logo {
    font-size: 48px;
    line-height: 1;
}
.app-header .titles h1 {
    font-family: 'Playfair Display', serif;
    font-size: 2rem;
    font-weight: 700;
    color: var(--txt-1);
    margin: 0 0 4px;
    letter-spacing: -0.5px;
}
.app-header .titles p {
    font-size: 0.85rem;
    color: var(--txt-2);
    margin: 0;
    font-weight: 300;
    letter-spacing: 0.3px;
}

/* ── Cards ────────────────────────────────────────────────── */
.card {
    background: var(--bg-2);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    padding: 24px;
    margin-bottom: 18px;
    box-shadow: var(--shadow);
}
.card-tight { padding: 16px; }

/* ── Section label ────────────────────────────────────────── */
.section-label {
    font-size: 0.7rem;
    font-weight: 600;
    letter-spacing: 1.8px;
    text-transform: uppercase;
    color: var(--txt-3);
    margin-bottom: 10px;
}

/* ── Verdict banner ───────────────────────────────────────── */
.verdict-normal {
    background: linear-gradient(135deg,rgba(0,212,170,.15),rgba(0,212,170,.05));
    border: 1px solid rgba(0,212,170,.35);
    border-radius: var(--radius);
    padding: 20px 24px;
    margin-bottom: 20px;
}
.verdict-pneumonia {
    background: linear-gradient(135deg,rgba(255,78,106,.15),rgba(255,78,106,.05));
    border: 1px solid rgba(255,78,106,.35);
    border-radius: var(--radius);
    padding: 20px 24px;
    margin-bottom: 20px;
}
.verdict-title {
    font-family: 'Playfair Display', serif;
    font-size: 1.5rem;
    font-weight: 700;
    margin: 0 0 4px;
}
.verdict-sub {
    font-size: 0.82rem;
    color: var(--txt-2);
    margin: 0;
}

/* ── Confidence bar ───────────────────────────────────────── */
.conf-bar-wrap {
    background: var(--bg-3);
    border-radius: 99px;
    height: 10px;
    overflow: hidden;
    margin: 8px 0 4px;
}
.conf-bar-fill-normal    { background: var(--accent2); border-radius:99px; height:10px; transition:width .6s ease; }
.conf-bar-fill-pneumonia { background: var(--danger);  border-radius:99px; height:10px; transition:width .6s ease; }

/* ── Severity badge ───────────────────────────────────────── */
.badge {
    display: inline-block;
    font-size: 0.7rem;
    font-weight: 600;
    letter-spacing: 0.8px;
    text-transform: uppercase;
    padding: 4px 10px;
    border-radius: 99px;
}
.badge-low    { background:rgba(0,212,170,.18); color:var(--accent2); }
.badge-moderate{ background:rgba(255,179,71,.18); color:var(--warn); }
.badge-high   { background:rgba(255,78,106,.18); color:var(--danger); }

/* ── Finding rows ─────────────────────────────────────────── */
.finding-row {
    display: flex;
    gap: 12px;
    align-items: flex-start;
    padding: 10px 0;
    border-bottom: 1px solid var(--border);
    font-size: 0.88rem;
}
.finding-row:last-child { border-bottom: none; }
.finding-icon { font-size: 1.1rem; flex-shrink:0; margin-top:1px; }
.finding-label { color: var(--txt-2); font-size:0.75rem; text-transform:uppercase; letter-spacing:.8px; margin-bottom:2px; }
.finding-val   { color: var(--txt-1); line-height:1.45; }

/* ── List bullets ─────────────────────────────────────────── */
.tip-list { list-style:none; padding:0; margin:0; }
.tip-list li {
    padding: 7px 0;
    border-bottom: 1px solid var(--border);
    font-size: 0.87rem;
    display: flex;
    gap: 8px;
    align-items: flex-start;
    line-height: 1.4;
}
.tip-list li:last-child { border-bottom:none; }

/* ── History entry ────────────────────────────────────────── */
.hist-entry {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 8px 0;
    border-bottom: 1px solid var(--border);
    font-size: 0.82rem;
}
.hist-entry:last-child { border-bottom: none; }
.hist-tag-normal    { color:var(--accent2); font-weight:600; }
.hist-tag-pneumonia { color:var(--danger);  font-weight:600; }

/* ── Mono chip ────────────────────────────────────────────── */
.mono { font-family:'DM Mono',monospace; font-size:0.82rem; color:var(--accent); }

/* ── Sidebar ──────────────────────────────────────────────── */
section[data-testid="stSidebar"] {
    background: var(--bg-1) !important;
    border-right: 1px solid var(--border) !important;
}

/* ── Disclaimer ───────────────────────────────────────────── */
.disclaimer {
    background: rgba(255,179,71,.07);
    border: 1px solid rgba(255,179,71,.25);
    border-radius: var(--radius);
    padding: 14px 18px;
    font-size: 0.78rem;
    color: var(--txt-2);
    line-height: 1.55;
}

/* ── Streamlit widget overrides ───────────────────────────── */
.stButton > button {
    background: var(--accent) !important;
    color: #fff !important;
    border: none !important;
    border-radius: 8px !important;
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 600 !important;
    letter-spacing: .3px !important;
    padding: 10px 26px !important;
    transition: opacity .2s !important;
}
.stButton > button:hover { opacity:.85 !important; }

.stFileUploader > div {
    background: var(--bg-2) !important;
    border: 2px dashed var(--border) !important;
    border-radius: var(--radius) !important;
}

/* hide default streamlit chrome */
#MainMenu, footer, header { visibility: hidden; }
</style>
""", unsafe_allow_html=True)


# ─────────────────────────────────────────────────────────────
# SESSION STATE
# ─────────────────────────────────────────────────────────────
if "history" not in st.session_state:
    st.session_state.history = []
if "last_result" not in st.session_state:
    st.session_state.last_result = None


# ─────────────────────────────────────────────────────────────
# MODEL LOADER  (cached for performance)
# ─────────────────────────────────────────────────────────────
@st.cache_resource(show_spinner=False)
def load_cnn_model(path: str = "/content/medical_model.h5"):
    """Load and cache the Keras CNN model."""
    return load_model(path)


# ─────────────────────────────────────────────────────────────
# IMAGE PREPROCESSING
# ─────────────────────────────────────────────────────────────
def preprocess_image(pil_image: Image.Image, target_size: tuple = (224, 224)) -> np.ndarray:
    """Convert PIL image → normalised numpy array for model input."""
    img = np.array(pil_image.convert("RGB"))
    img = cv2.resize(img, target_size)
    img = img / 255.0
    return np.expand_dims(img, axis=0)   # (1, H, W, 3)


# ─────────────────────────────────────────────────────────────
# GRAD-CAM  (optional — skipped gracefully if model lacks conv layers)
# ─────────────────────────────────────────────────────────────
def generate_gradcam(model, img_array: np.ndarray, target_size: tuple = (224, 224)) -> np.ndarray | None:
    """
    Generate a Grad-CAM heatmap overlaid on the original image.
    Returns an RGB numpy array, or None if the model has no Conv layers.
    """
    try:
        # Find last conv layer
        last_conv = None
        for layer in reversed(model.layers):
            if isinstance(layer, (
                __import__("tensorflow").keras.layers.Conv2D,
            )):
                last_conv = layer.name
                break
        if last_conv is None:
            return None

        grad_model = Model(
            inputs=model.inputs,
            outputs=[model.get_layer(last_conv).output, model.output]
        )

        import tensorflow as tf
        with tf.GradientTape() as tape:
            conv_outs, preds = grad_model(img_array)
            loss = preds[:, 0]

        grads = tape.gradient(loss, conv_outs)[0]
        conv_outs = conv_outs[0]
        weights = tf.reduce_mean(grads, axis=(0, 1))
        cam = np.zeros(conv_outs.shape[:2], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * conv_outs[:, :, i]

        cam = np.maximum(cam, 0)
        cam /= cam.max() + 1e-8
        cam = cv2.resize(cam.numpy(), target_size)

        # Overlay on original image
        orig = (img_array[0] * 255).astype(np.uint8)
        heatmap = np.uint8(255 * cm.jet(cam)[:, :, :3])
        overlay = cv2.addWeighted(orig, 0.55, heatmap, 0.45, 0)
        return overlay
    except Exception:
        return None


# ─────────────────────────────────────────────────────────────
# PREDICTION
# ─────────────────────────────────────────────────────────────
def run_prediction(model, img_array: np.ndarray) -> tuple[str, float]:
    """Return (label, confidence_pct)."""
    raw = model.predict(img_array, verbose=0)[0][0]
    if raw > 0.5:
        return "Pneumonia", float(raw) * 100
    return "Normal", float(1 - raw) * 100


def severity_label(conf: float, result: str) -> str:
    """Map confidence + result to severity string."""
    if result == "Normal":
        return "Low Risk"
    if conf < 70:
        return "Moderate"
    return "High"


# ─────────────────────────────────────────────────────────────
# REPORT DATA  (structured knowledge base)
# ─────────────────────────────────────────────────────────────
REPORT_KB = {
    "Normal": {
        "summary": "No pathological findings detected.",
        "headline": "Chest X-ray — Normal",
        "findings": [
            ("🫁", "Lung Fields", "Clear bilaterally. No consolidation, effusion, or infiltrate observed."),
            ("💓", "Cardiac Silhouette", "Normal size and contour. Cardiothoracic ratio within limits."),
            ("🦴", "Bony Structures", "Ribs, clavicles, and spine appear intact with no visible fractures."),
            ("🌬️", "Diaphragm", "Bilateral domes well-defined. No sub-phrenic free air."),
        ],
        "patient_explanation": (
            "Your chest X-ray shows no signs of infection, fluid, or abnormal tissue. "
            "The lungs appear clear and the heart size is normal. This is a healthy result."
        ),
        "possible_causes": [],
        "do_tips": [
            ("🥦", "Eat a balanced, nutrient-rich diet"),
            ("🚶", "Exercise at least 30 min, 5 days a week"),
            ("💧", "Stay well-hydrated (8+ glasses of water daily)"),
            ("😴", "Get 7–9 hours of quality sleep each night"),
            ("🩺", "Schedule annual preventive health screenings"),
        ],
        "avoid_tips": [
            ("🚬", "Smoking or secondhand smoke exposure"),
            ("🏭", "Prolonged exposure to air pollutants and dust"),
            ("🧴", "Unnecessary use of respiratory irritants"),
        ],
        "recovery_tips": [],
        "doctor_advice": "No urgent consultation required. Continue routine annual check-ups.",
        "follow_up": "12 months",
    },
    "Pneumonia": {
        "summary": "Pulmonary opacity consistent with pneumonia detected.",
        "headline": "Chest X-ray — Pneumonia Detected",
        "findings": [
            ("⚠️", "Lung Opacity", "Patchy or lobar consolidation present, suggesting alveolar infiltration."),
            ("💧", "Possible Effusion", "Small pleural effusion may be present adjacent to the affected lobe."),
            ("🌡️", "Inflammatory Signs", "Increased radio-opacity in affected segments consistent with infection."),
            ("💓", "Cardiac Silhouette", "Borders may be partially obscured by adjacent consolidation."),
        ],
        "patient_explanation": (
            "Your X-ray shows areas of cloudiness in one or both lungs. This is typically caused by "
            "bacteria, viruses, or fungi filling the air sacs with fluid or pus, making it harder to breathe. "
            "Prompt medical attention is strongly advised."
        ),
        "possible_causes": [
            ("🦠", "Bacterial (e.g., Streptococcus pneumoniae — most common)"),
            ("🧬", "Viral (e.g., influenza, COVID-19, RSV)"),
            ("🍄", "Fungal (less common, usually immunocompromised patients)"),
            ("🌬️", "Aspiration of food or liquid into the lungs"),
        ],
        "do_tips": [
            ("💊", "Take all prescribed antibiotics / antivirals as directed"),
            ("🛌", "Rest completely — avoid strenuous activity"),
            ("💧", "Increase fluid intake to thin mucus secretions"),
            ("🌡️", "Monitor temperature; seek ER if fever > 39.5°C / 103°F"),
            ("🩺", "Follow up with your physician within 48–72 hours"),
        ],
        "avoid_tips": [
            ("🚬", "Smoking — severely impairs lung healing"),
            ("🥶", "Cold, damp environments that worsen respiratory symptoms"),
            ("🍺", "Alcohol — suppresses immune response"),
            ("🏃", "Heavy physical exertion until fully recovered"),
            ("😷", "Contact with immunocompromised individuals while infectious"),
        ],
        "recovery_tips": [
            ("📅", "Mild cases: 1–3 weeks recovery with antibiotics"),
            ("🏥", "Severe cases may require hospitalisation and IV antibiotics"),
            ("💉", "Consider pneumococcal and flu vaccines post-recovery"),
            ("🌬️", "Breathing exercises to restore full lung capacity"),
        ],
        "doctor_advice": "Consult a physician immediately. Do not delay — pneumonia can progress rapidly.",
        "follow_up": "48–72 hours",
    },
}


# ─────────────────────────────────────────────────────────────
# REPORT RENDERER
# ─────────────────────────────────────────────────────────────
def render_report(result: str, conf: float, gradcam_img: np.ndarray | None = None):
    """Render the full structured medical report in the right column."""
    data = REPORT_KB[result]
    sev  = severity_label(conf, result)

    verdict_class = "verdict-normal" if result == "Normal" else "verdict-pneumonia"
    icon  = "✅" if result == "Normal" else "🚨"
    color = "#00d4aa" if result == "Normal" else "#ff4e6a"
    bar_class = "conf-bar-fill-normal" if result == "Normal" else "conf-bar-fill-pneumonia"
    badge_cls = "badge-low" if result == "Normal" else ("badge-moderate" if sev == "Moderate" else "badge-high")

    # ── Verdict ──────────────────────────────────────────────
    st.markdown(f"""
    <div class="{verdict_class}">
      <div class="verdict-title" style="color:{color};">{icon} {data['headline']}</div>
      <p class="verdict-sub">{data['summary']}</p>
      <div style="margin-top:12px;">
        <div style="display:flex;justify-content:space-between;font-size:0.78rem;color:var(--txt-2);margin-bottom:4px;">
          <span>Confidence Score</span>
          <span class="mono">{conf:.1f}%</span>
        </div>
        <div class="conf-bar-wrap">
          <div class="{bar_class}" style="width:{conf:.1f}%;"></div>
        </div>
        <div style="margin-top:8px;">
          <span class="badge {badge_cls}">Severity: {sev}</span>
          <span style="font-size:0.75rem;color:var(--txt-3);margin-left:12px;">
            Follow-up: {data['follow_up']}
          </span>
        </div>
      </div>
    </div>
    """, unsafe_allow_html=True)

    # ── Key Findings ─────────────────────────────────────────
    with st.expander("🔬 Key Radiological Findings", expanded=True):
        rows_html = ""
        for icon_f, label, val in data["findings"]:
            rows_html += f"""
            <div class="finding-row">
              <div class="finding-icon">{icon_f}</div>
              <div>
                <div class="finding-label">{label}</div>
                <div class="finding-val">{val}</div>
              </div>
            </div>"""
        st.markdown(f'<div class="card card-tight">{rows_html}</div>', unsafe_allow_html=True)

    # ── Patient Explanation ───────────────────────────────────
    with st.expander("🗣️ Plain-Language Explanation", expanded=True):
        st.markdown(f"""
        <div class="card card-tight">
          <p style="font-size:0.9rem;color:var(--txt-1);line-height:1.65;margin:0;">
            {data['patient_explanation']}
          </p>
        </div>""", unsafe_allow_html=True)

    # ── Possible Causes (Pneumonia only) ─────────────────────
    if data["possible_causes"]:
        with st.expander("🦠 Possible Causes"):
            items = "".join(
                f'<li><span>{ic}</span><span>{txt}</span></li>'
                for ic, txt in data["possible_causes"]
            )
            st.markdown(f'<ul class="tip-list">{items}</ul>', unsafe_allow_html=True)

    # ── Recommendations ───────────────────────────────────────
    col_a, col_b = st.columns(2)
    with col_a:
        with st.expander("✅ What To Do"):
            items = "".join(f'<li><span>{ic}</span><span>{txt}</span></li>' for ic, txt in data["do_tips"])
            st.markdown(f'<ul class="tip-list">{items}</ul>', unsafe_allow_html=True)
    with col_b:
        with st.expander("❌ What To Avoid"):
            items = "".join(f'<li><span>{ic}</span><span>{txt}</span></li>' for ic, txt in data["avoid_tips"])
            st.markdown(f'<ul class="tip-list">{items}</ul>', unsafe_allow_html=True)

    # ── Recovery Tips (Pneumonia only) ───────────────────────
    if data["recovery_tips"]:
        with st.expander("🔄 Recovery & Follow-Up"):
            items = "".join(f'<li><span>{ic}</span><span>{txt}</span></li>' for ic, txt in data["recovery_tips"])
            st.markdown(f'<ul class="tip-list">{items}</ul>', unsafe_allow_html=True)

    # ── Doctor Recommendation ─────────────────────────────────
    bg = "rgba(255,78,106,.08)" if result == "Pneumonia" else "rgba(0,212,170,.08)"
    bc = "rgba(255,78,106,.3)"  if result == "Pneumonia" else "rgba(0,212,170,.3)"
    st.markdown(f"""
    <div style="background:{bg};border:1px solid {bc};border-radius:var(--radius);padding:14px 18px;margin-top:8px;">
      <div style="font-size:0.72rem;letter-spacing:1.2px;text-transform:uppercase;color:var(--txt-3);margin-bottom:6px;">
        👨‍⚕️ Doctor Recommendation
      </div>
      <p style="margin:0;font-size:0.88rem;color:var(--txt-1);">{data['doctor_advice']}</p>
    </div>""", unsafe_allow_html=True)

    # ── Grad-CAM Heatmap ──────────────────────────────────────
    if gradcam_img is not None:
        with st.expander("🌡️ Grad-CAM Heatmap — Attention Regions"):
            st.markdown("""
            <p style="font-size:0.8rem;color:var(--txt-2);margin-bottom:10px;">
            Red/yellow regions indicate areas the model weighted most heavily for its decision.
            This is for research purposes only and not a clinical annotation.
            </p>""", unsafe_allow_html=True)
            fig, ax = plt.subplots(figsize=(5, 5))
            fig.patch.set_facecolor("#111827")
            ax.imshow(gradcam_img)
            ax.axis("off")
            st.pyplot(fig, use_container_width=False)
            plt.close(fig)


# ─────────────────────────────────────────────────────────────
# REPORT DOWNLOAD TEXT GENERATOR
# ─────────────────────────────────────────────────────────────
def generate_report_text(result: str, conf: float, timestamp: str) -> str:
    data = REPORT_KB[result]
    sev  = severity_label(conf, result)
    lines = [
        "=" * 62,
        "   PNEUMOSCAN AI — MEDICAL IMAGING REPORT",
        "=" * 62,
        f"  Date/Time  : {timestamp}",
        f"  Diagnosis  : {result}",
        f"  Confidence : {conf:.2f}%",
        f"  Severity   : {sev}",
        f"  Follow-up  : {data['follow_up']}",
        "",
        "SUMMARY",
        "-" * 40,
        data["summary"],
        "",
        "RADIOLOGICAL FINDINGS",
        "-" * 40,
    ]
    for _, label, val in data["findings"]:
        lines.append(f"  {label}: {val}")
    lines += [
        "",
        "PATIENT EXPLANATION",
        "-" * 40,
        data["patient_explanation"],
        "",
    ]
    if data["possible_causes"]:
        lines += ["POSSIBLE CAUSES", "-" * 40]
        for _, txt in data["possible_causes"]:
            lines.append(f"  • {txt}")
        lines.append("")
    lines += ["RECOMMENDATIONS — WHAT TO DO", "-" * 40]
    for _, txt in data["do_tips"]:
        lines.append(f"  ✔ {txt}")
    lines += ["", "RECOMMENDATIONS — WHAT TO AVOID", "-" * 40]
    for _, txt in data["avoid_tips"]:
        lines.append(f"  ✘ {txt}")
    if data["recovery_tips"]:
        lines += ["", "RECOVERY & FOLLOW-UP", "-" * 40]
        for _, txt in data["recovery_tips"]:
            lines.append(f"  → {txt}")
    lines += [
        "",
        "DOCTOR RECOMMENDATION",
        "-" * 40,
        data["doctor_advice"],
        "",
        "=" * 62,
        "⚠  DISCLAIMER",
        "=" * 62,
        "This report is generated by an AI model for educational",
        "and screening purposes ONLY. It is NOT a substitute for",
        "a qualified medical professional's diagnosis. Always",
        "consult a licensed physician before making any health",
        "decisions.",
        "=" * 62,
    ]
    return "\n".join(lines)


# ─────────────────────────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────────────────────────
def render_sidebar():
    with st.sidebar:
        st.markdown("""
        <div style="padding:16px 0 8px;">
          <div style="font-size:1.05rem;font-weight:700;color:var(--txt-1);letter-spacing:-.3px;">VisionDx AI</div>
          <div style="font-size:0.72rem;color:var(--txt-3);margin-top:2px;">v2.0 · Clinical Preview</div>
        </div>
        <hr style="border-color:var(--border);margin:8px 0 16px;">
        """, unsafe_allow_html=True)

        # ── History ──────────────────────────────────────────
        st.markdown('<div class="section-label">📋 Scan History</div>', unsafe_allow_html=True)

        if st.session_state.history:
            entries_html = ""
            for h in reversed(st.session_state.history[-10:]):
                tag_cls = "hist-tag-normal" if h["result"] == "Normal" else "hist-tag-pneumonia"
                entries_html += f"""
                <div class="hist-entry">
                  <span style="color:var(--txt-3);">{h['time']}</span>
                  <span class="{tag_cls}">{h['result']}</span>
                  <span class="mono">{h['conf']:.1f}%</span>
                </div>"""
            st.markdown(f'<div class="card card-tight">{entries_html}</div>', unsafe_allow_html=True)

            if st.button("🗑 Clear History"):
                st.session_state.history = []
                st.session_state.last_result = None
                st.rerun()
        else:
            st.markdown('<p style="font-size:0.8rem;color:var(--txt-3);">No scans yet.</p>', unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)

        # ── Guidelines ───────────────────────────────────────
        with st.expander("📖 Image Guidelines"):
            st.markdown("""
            <ul class="tip-list">
              <li><span>✅</span><span>PA or AP chest X-ray only</span></li>
              <li><span>✅</span><span>Clear, high-contrast image</span></li>
              <li><span>✅</span><span>Formats: JPG, PNG, JPEG</span></li>
              <li><span>❌</span><span>Blurry or low-resolution images</span></li>
              <li><span>❌</span><span>Non-chest or unrelated photos</span></li>
              <li><span>❌</span><span>Heavily filtered images</span></li>
            </ul>""", unsafe_allow_html=True)

        # ── About ─────────────────────────────────────────────
        with st.expander("ℹ️ About This Tool"):
            st.markdown("""
            <p style="font-size:0.8rem;color:var(--txt-2);line-height:1.6;margin:0;">
            PneumoScan AI uses a convolutional neural network trained on the
            <strong style="color:var(--txt-1);">Kaggle Chest X-Ray Images</strong> dataset
            (5,863 images) to distinguish Normal from Pneumonia findings.<br><br>
            Model architecture: Custom CNN / Transfer Learning (VGG16 / ResNet base).<br>
            Accuracy on test set: <strong style="color:var(--accent);">~90–95%</strong> (varies by model).
            </p>""", unsafe_allow_html=True)

        # ── Disclaimer ───────────────────────────────────────
        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown("""
        <div class="disclaimer">
          ⚠️ <strong>Medical Disclaimer</strong><br>
          This tool is for educational and screening purposes only.
          It does <em>not</em> replace professional medical advice,
          diagnosis, or treatment. Always consult a qualified
          physician with any health concerns.
        </div>""", unsafe_allow_html=True)


# ─────────────────────────────────────────────────────────────
# MAIN APP
# ─────────────────────────────────────────────────────────────
def main():
    render_sidebar()

    # ── Page-load deep breath animation (auto-dismisses after ~3.5s) ──
    st.markdown("""
    <style>
    /* Overlay */
    #breath-overlay {
        position: fixed;
        inset: 0;
        z-index: 99999;
        background: #0a0e1a;
        display: flex;
        flex-direction: column;
        align-items: center;
        justify-content: center;
        gap: 28px;
        animation: overlayFadeOut 0.6s ease 3.4s forwards;
        pointer-events: none;
    }
    @keyframes overlayFadeOut {
        to { opacity: 0; visibility: hidden; }
    }

    /* Lung SVG pulse */
    .breath-lung {
        width: 110px;
        height: 110px;
        animation: lungBreath 3.2s ease-in-out 1 forwards;
        filter: drop-shadow(0 0 22px rgba(59,158,255,0.45));
    }
    @keyframes lungBreath {
        0%   { transform: scale(0.72); opacity: 0.3; }
        30%  { transform: scale(1.0);  opacity: 1;   }   /* inhale */
        55%  { transform: scale(1.18); opacity: 1;   }   /* hold   */
        80%  { transform: scale(0.88); opacity: 0.85;}   /* exhale */
        100% { transform: scale(0.72); opacity: 0.2; }
    }

    /* Ripple rings */
    .breath-rings {
        position: absolute;
        width: 110px;
        height: 110px;
    }
    .breath-rings span {
        position: absolute;
        inset: 0;
        border-radius: 50%;
        border: 2px solid rgba(59,158,255,0.25);
        animation: ringExpand 3.2s ease-out 1 forwards;
    }
    .breath-rings span:nth-child(2) { animation-delay: 0.4s; }
    .breath-rings span:nth-child(3) { animation-delay: 0.8s; }
    @keyframes ringExpand {
        0%   { transform: scale(1);   opacity: 0.6; }
        100% { transform: scale(2.8); opacity: 0;   }
    }

    /* Text */
    .breath-text {
        font-family: 'DM Sans', sans-serif;
        font-size: 0.82rem;
        letter-spacing: 2.5px;
        text-transform: uppercase;
        color: rgba(143,163,190,0.7);
        animation: textPulse 3.2s ease-in-out 1 forwards;
    }
    @keyframes textPulse {
        0%,100% { opacity: 0.3; }
        40%      { opacity: 1;   }
    }

    /* Thin progress line */
    .breath-line {
        width: 160px;
        height: 2px;
        background: rgba(59,158,255,0.12);
        border-radius: 99px;
        overflow: hidden;
    }
    .breath-line-fill {
        height: 100%;
        width: 0%;
        background: linear-gradient(90deg, #3b9eff, #00d4aa);
        border-radius: 99px;
        animation: lineGrow 3.2s ease-in-out 1 forwards;
    }
    @keyframes lineGrow {
        0%   { width: 0%; }
        55%  { width: 75%; }
        100% { width: 100%; }
    }
    </style>

    <div id="breath-overlay">
      <div style="position:relative;display:flex;align-items:center;justify-content:center;">
        <div class="breath-rings">
          <span></span><span></span><span></span>
        </div>
        <svg class="breath-lung" viewBox="0 0 100 100" xmlns="http://www.w3.org/2000/svg">
          <!-- left lobe -->
          <path d="M50 30 C50 30 36 28 28 36 C18 46 18 62 24 72
                   C28 79 36 82 42 78 C46 75 48 68 48 60 L48 30 Z"
                fill="none" stroke="#3b9eff" stroke-width="2.2"
                stroke-linecap="round" stroke-linejoin="round"
                opacity="0.9"/>
          <!-- right lobe -->
          <path d="M50 30 C50 30 64 28 72 36 C82 46 82 62 76 72
                   C72 79 64 82 58 78 C54 75 52 68 52 60 L52 30 Z"
                fill="none" stroke="#00d4aa" stroke-width="2.2"
                stroke-linecap="round" stroke-linejoin="round"
                opacity="0.9"/>
          <!-- trachea -->
          <line x1="50" y1="14" x2="50" y2="32"
                stroke="#8fa3be" stroke-width="2" stroke-linecap="round"/>
          <!-- bronchi -->
          <path d="M50 32 C50 32 44 34 42 38" stroke="#8fa3be" stroke-width="1.6"
                fill="none" stroke-linecap="round"/>
          <path d="M50 32 C50 32 56 34 58 38" stroke="#8fa3be" stroke-width="1.6"
                fill="none" stroke-linecap="round"/>
          <!-- inner fill glow -->
          <path d="M50 30 C50 30 36 28 28 36 C18 46 18 62 24 72
                   C28 79 36 82 42 78 C46 75 48 68 48 60 L48 30 Z"
                fill="rgba(59,158,255,0.06)"/>
          <path d="M50 30 C50 30 64 28 72 36 C82 46 82 62 76 72
                   C72 79 64 82 58 78 C54 75 52 68 52 60 L52 30 Z"
                fill="rgba(0,212,170,0.06)"/>
        </svg>
      </div>
      <div class="breath-text">Initialising · VisionDx AI</div>
      <div class="breath-line"><div class="breath-line-fill"></div></div>
    </div>
    """, unsafe_allow_html=True)

    # ── App Header ───────────────────────────────────────────
    st.markdown("""
    <style>
    /* ── Logo container ────────────────────────────────── */
    .vdx-header {
        display: flex;
        align-items: center;
        gap: 22px;
        padding: 24px 0 20px;
        border-bottom: 1px solid var(--border);
        margin-bottom: 28px;
        animation: headerSlideIn 0.7s cubic-bezier(.22,.68,0,1.2) 3.6s both;
    }
    @keyframes headerSlideIn {
        from { opacity:0; transform:translateY(-18px); }
        to   { opacity:1; transform:translateY(0);     }
    }

    /* ── SVG eye+scan icon ─────────────────────────────── */
    .vdx-logo-svg {
        flex-shrink: 0;
        width: 64px;
        height: 64px;
        filter: drop-shadow(0 0 14px rgba(59,158,255,0.5))
                drop-shadow(0 0 28px rgba(0,212,170,0.2));
        animation: logoPop 0.6s cubic-bezier(.34,1.56,.64,1) 3.7s both;
    }
    @keyframes logoPop {
        from { transform: scale(0.5) rotate(-15deg); opacity:0; }
        to   { transform: scale(1)   rotate(0deg);   opacity:1; }
    }

    /* Iris scan ring spin */
    .vdx-scan-ring {
        transform-origin: 32px 32px;
        animation: scanSpin 8s linear infinite;
    }
    @keyframes scanSpin {
        to { transform: rotate(360deg); }
    }
    /* Reverse ring */
    .vdx-scan-ring-rev {
        transform-origin: 32px 32px;
        animation: scanSpinRev 6s linear infinite;
    }
    @keyframes scanSpinRev {
        to { transform: rotate(-360deg); }
    }

    /* Pupil pulse */
    .vdx-pupil {
        transform-origin: 32px 32px;
        animation: pupilPulse 2.4s ease-in-out infinite;
    }
    @keyframes pupilPulse {
        0%,100% { transform: scale(1);    }
        50%      { transform: scale(1.18); }
    }

    /* Scan line sweep */
    .vdx-scanline {
        animation: scanLine 2.8s ease-in-out infinite;
        transform-origin: 32px 32px;
    }
    @keyframes scanLine {
        0%   { opacity:0; transform:translateY(-14px); }
        30%  { opacity:1; }
        70%  { opacity:1; }
        100% { opacity:0; transform:translateY(14px);  }
    }

    /* ── Brand name ─────────────────────────────────────── */
    .vdx-brand-wrap { overflow: hidden; }

    .vdx-brand-name {
        font-family: 'Playfair Display', serif;
        font-size: 2.1rem;
        font-weight: 700;
        letter-spacing: -0.5px;
        line-height: 1.1;
        margin: 0 0 5px;
        display: flex;
        align-items: baseline;
        gap: 0px;
        animation: brandReveal 0.8s cubic-bezier(.22,.68,0,1.1) 3.8s both;
    }
    @keyframes brandReveal {
        from { opacity:0; transform:translateX(-20px); }
        to   { opacity:1; transform:translateX(0);     }
    }

    /* "Vision" — gradient text */
    .vdx-word-vision {
        background: linear-gradient(90deg, #3b9eff 0%, #00d4aa 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        background-clip: text;
    }
    /* "Dx" — teal accent */
    .vdx-word-dx {
        color: #00d4aa;
        font-style: italic;
        margin: 0 4px;
    }
    /* "AI" — subtle */
    .vdx-word-ai {
        color: var(--txt-2);
        font-size: 1.5rem;
        font-weight: 400;
        font-family: 'DM Mono', monospace;
        letter-spacing: 1px;
    }

    /* Blinking cursor after AI */
    .vdx-cursor {
        display: inline-block;
        width: 3px;
        height: 1.5rem;
        background: var(--accent);
        border-radius: 2px;
        margin-left: 5px;
        vertical-align: middle;
        animation: cursorBlink 1.1s step-end infinite 4.4s both;
        opacity: 0;
    }
    @keyframes cursorBlink {
        0%,100% { opacity:1; }
        50%      { opacity:0; }
    }

    /* Subtitle */
    .vdx-subtitle {
        font-size: 0.83rem;
        color: var(--txt-3);
        font-weight: 300;
        letter-spacing: 0.4px;
        margin: 0;
        animation: brandReveal 0.8s cubic-bezier(.22,.68,0,1.1) 4.0s both;
    }

    /* Live badge */
    .vdx-live-badge {
        display: inline-flex;
        align-items: center;
        gap: 5px;
        background: rgba(0,212,170,0.1);
        border: 1px solid rgba(0,212,170,0.3);
        border-radius: 99px;
        padding: 2px 9px;
        font-size: 0.66rem;
        font-weight: 600;
        letter-spacing: 1px;
        text-transform: uppercase;
        color: #00d4aa;
        margin-left: 12px;
        vertical-align: middle;
        animation: brandReveal 0.6s ease 4.2s both;
    }
    .vdx-dot {
        width: 6px; height: 6px;
        border-radius: 50%;
        background: #00d4aa;
        animation: dotPulse 1.4s ease-in-out infinite;
    }
    @keyframes dotPulse {
        0%,100% { opacity:1; transform:scale(1);    }
        50%      { opacity:.4; transform:scale(0.7); }
    }
    </style>

    <div class="vdx-header">

      <!-- ── Animated Eye + Scan SVG logo ── -->
      <svg class="vdx-logo-svg" viewBox="0 0 64 64" fill="none" xmlns="http://www.w3.org/2000/svg">
        <!-- Outer glow ring (spinning dashes) -->
        <circle class="vdx-scan-ring"
                cx="32" cy="32" r="29"
                stroke="rgba(59,158,255,0.25)" stroke-width="1.2"
                stroke-dasharray="6 4" stroke-linecap="round"/>
        <!-- Mid ring (reverse spin) -->
        <circle class="vdx-scan-ring-rev"
                cx="32" cy="32" r="23"
                stroke="rgba(0,212,170,0.2)" stroke-width="1"
                stroke-dasharray="3 6" stroke-linecap="round"/>
        <!-- Eye outline (eyelid curves) -->
        <path d="M 8 32 Q 20 14 32 14 Q 44 14 56 32 Q 44 50 32 50 Q 20 50 8 32 Z"
              fill="rgba(59,158,255,0.06)"
              stroke="#3b9eff" stroke-width="1.6" stroke-linejoin="round"/>
        <!-- Iris -->
        <circle cx="32" cy="32" r="10"
                fill="rgba(0,212,170,0.08)"
                stroke="#00d4aa" stroke-width="1.8"/>
        <!-- Pupil -->
        <circle class="vdx-pupil" cx="32" cy="32" r="5"
                fill="rgba(59,158,255,0.55)"
                stroke="#3b9eff" stroke-width="1"/>
        <!-- Pupil glint -->
        <circle cx="34.5" cy="29.5" r="1.4" fill="rgba(255,255,255,0.7)"/>
        <!-- Scan line sweeping vertically through eye -->
        <line class="vdx-scanline"
              x1="14" y1="32" x2="50" y2="32"
              stroke="rgba(0,212,170,0.7)" stroke-width="1.2"
              stroke-linecap="round"
              stroke-dasharray="4 3"/>
        <!-- Corner tick marks -->
        <path d="M8 26 L8 20 L14 20"  stroke="#3b9eff" stroke-width="1.4" fill="none" stroke-linecap="round"/>
        <path d="M56 26 L56 20 L50 20" stroke="#3b9eff" stroke-width="1.4" fill="none" stroke-linecap="round"/>
        <path d="M8 38 L8 44 L14 44"  stroke="#00d4aa" stroke-width="1.4" fill="none" stroke-linecap="round"/>
        <path d="M56 38 L56 44 L50 44" stroke="#00d4aa" stroke-width="1.4" fill="none" stroke-linecap="round"/>
      </svg>

      <!-- ── Brand name + subtitle ── -->
      <div class="vdx-brand-wrap">
        <div class="vdx-brand-name">
          <span class="vdx-word-vision">Vision</span>
          <span class="vdx-word-dx">Dx</span>
          <span class="vdx-word-ai">&nbsp;AI</span>
          <span class="vdx-cursor"></span>
          <span class="vdx-live-badge"><span class="vdx-dot"></span>Live</span>
        </div>
        <p class="vdx-subtitle">
          AI-Powered Chest X-Ray Analysis &nbsp;·&nbsp; Pneumonia Detection System
        </p>
      </div>

    </div>
    """, unsafe_allow_html=True)

    # ── Layout columns ───────────────────────────────────────
    left, right = st.columns([1, 1.3], gap="large")

    # ── LEFT: Upload & Preview ───────────────────────────────
    with left:
        st.markdown('<div class="section-label">📤 Upload X-Ray Image</div>', unsafe_allow_html=True)
        uploaded = st.file_uploader(
            label="Drop your chest X-ray here",
            type=["jpg", "jpeg", "png"],
            label_visibility="collapsed",
        )

        if uploaded:
            pil_img = Image.open(uploaded).convert("RGB")
            st.image(pil_img, caption="Uploaded X-Ray", use_container_width=True)

            # Image metadata card
            w, h = pil_img.size
            file_kb = uploaded.size / 1024
            st.markdown(f"""
            <div class="card card-tight" style="margin-top:12px;">
              <div class="section-label">Image Info</div>
              <div style="display:flex;gap:24px;font-size:0.82rem;color:var(--txt-2);">
                <div><span class="mono">{w}×{h}</span> px</div>
                <div><span class="mono">{file_kb:.1f}</span> KB</div>
                <div><span class="mono">{uploaded.type}</span></div>
              </div>
            </div>""", unsafe_allow_html=True)

            # ── Analyse button ────────────────────────────────
            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("🔍  Analyse X-Ray", use_container_width=True):
                # Load model
                with st.spinner("Loading AI model…"):
                    try:
                        model = load_cnn_model()
                    except Exception as e:
                        st.error(f"❌ Could not load model: {e}")
                        return

                # Preprocessing + prediction with animated progress
                progress_bar = st.progress(0, text="Preprocessing image…")
                img_array = preprocess_image(pil_img)
                for p in range(0, 40, 5):
                    time.sleep(0.04)
                    progress_bar.progress(p, text="Preprocessing image…")

                progress_bar.progress(40, text="Running inference…")
                result, conf = run_prediction(model, img_array)
                for p in range(40, 80, 5):
                    time.sleep(0.04)
                    progress_bar.progress(p, text="Running inference…")

                progress_bar.progress(80, text="Generating Grad-CAM…")
                gradcam = generate_gradcam(model, img_array)
                for p in range(80, 100, 5):
                    time.sleep(0.03)
                    progress_bar.progress(p, text="Compiling report…")
                progress_bar.progress(100, text="Done ✓")
                time.sleep(0.3)
                progress_bar.empty()

                # Save to history
                ts = datetime.datetime.now().strftime("%H:%M")
                st.session_state.history.append({"time": ts, "result": result, "conf": conf})
                st.session_state.last_result = {
                    "result": result,
                    "conf": conf,
                    "gradcam": gradcam,
                    "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                }
                st.rerun()
        else:
            st.markdown("""
            <div class="card" style="text-align:center;padding:60px 24px;border:2px dashed var(--border);">
              <div style="font-size:3rem;margin-bottom:12px;">🩻</div>
              <div style="color:var(--txt-2);font-size:0.9rem;">Upload a chest X-ray image<br>to begin AI analysis</div>
            </div>""", unsafe_allow_html=True)

    # ── RIGHT: Report ─────────────────────────────────────────
    with right:
        if st.session_state.last_result:
            lr = st.session_state.last_result
            st.markdown('<div class="section-label">📋 Medical Report</div>', unsafe_allow_html=True)
            render_report(lr["result"], lr["conf"], lr.get("gradcam"))

            # ── Download ──────────────────────────────────────
            report_txt = generate_report_text(lr["result"], lr["conf"], lr["timestamp"])
            st.download_button(
                label="📥  Download Full Report (.txt)",
                data=report_txt.encode("utf-8"),
                file_name=f"pneumoscan_report_{lr['timestamp'].replace(' ','_').replace(':','-')}.txt",
                mime="text/plain",
                use_container_width=True,
            )
        else:
            st.markdown("""
            <div class="card" style="text-align:center;padding:80px 24px;">
              <div style="font-size:3rem;margin-bottom:16px;">📋</div>
              <div style="color:var(--txt-2);font-size:0.9rem;line-height:1.6;">
                Your detailed medical report will appear here<br>after analysis is complete.
              </div>
            </div>""", unsafe_allow_html=True)

    # ── Footer disclaimer ─────────────────────────────────────
    st.markdown("<br>", unsafe_allow_html=True)
    st.markdown("""
    <div class="disclaimer" style="text-align:center;">
      ⚠️ <strong>For educational and research use only.</strong>
      This AI system does not constitute medical advice. Consult a licensed physician for diagnosis and treatment.
    </div>""", unsafe_allow_html=True)


if __name__ == "__main__":
    main()

Overwriting app.py


In [100]:
from pyngrok import ngrok

ngrok.set_auth_token("31sb4xzFoVlFcoRlzzbGxGJRXgy_6HAfkhmRKcMkU23uNMDHz")

In [101]:
from pyngrok import ngrok

ngrok.kill()

In [102]:
!streamlit run app.py &>/dev/null &

In [103]:
public_url = ngrok.connect(8501)
public_url

<NgrokTunnel: "https://37b1-35-227-147-23.ngrok-free.app" -> "http://localhost:8501">